# Lab 2.3 – Memory-Enabled AI Assistant

## Objective

The objective of this lab is to build an AI assistant that combines conversational memory with multiple custom tools.

The assistant can:

- Remember previous conversations.
- Select the appropriate tool.
- Execute the tool.
- Continue the conversation using previous context.

## Technologies Used

- Python
- LangChain
- Groq
- Jupyter Notebook

# Introduction

Modern AI assistants require both conversational memory and external tools.

Memory allows the assistant to remember previous interactions, while tools enable it to perform tasks such as calculations, searching databases, formatting dates, converting currencies, and summarizing text.

This lab combines both concepts into a single AI assistant.

# AI Assistant Architecture

```
User

 │
 ▼

Conversation Memory

 │
 ▼

Large Language Model
 │
 ▼

Choose Tool

 │
 ▼

Execute Tool

 │
 ▼

Store Response

 │
 ▼
Return Answer
```

In [1]:
from dotenv import load_dotenv
import os

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv()

True

In [2]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq LLM Initialized Successfully!")

Groq LLM Initialized Successfully!


# Creating Custom Tools

The tools created in Day 3 are reused in this lab.

For simplicity, copy the following tools from your Day 3 notebook:

- Calculator
- Weather
- Database Search
- Date Formatter
- Currency Converter
- Text Summarizer

In [3]:
@tool
def calculate(expression: str) -> str:
    """
    Evaluates a mathematical expression.
    """

    try:
        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Error: {e}"

In [4]:
@tool
def weather(city: str) -> str:
    """
    Returns the weather for a given city.
    """

    weather_data = {
        "Islamabad": "Sunny, 42°C",
        "Lahore": "Cloudy, 45°C",
        "Karachi": "Hot, 47°C",
        "Peshawar": "Sunny, 44°C"
    }

    return weather_data.get(city, "Weather data not available.")

In [5]:
@tool
def search_db(topic: str) -> str:
    """
    Searches a simple database.
    """

    database = {
        "python": "Python is a high-level programming language.",
        "langchain": "LangChain is a framework for LLM applications.",
        "groq": "Groq provides fast inference for Large Language Models.",
        "pydantic": "Pydantic validates structured data using Python type hints."
    }

    return database.get(topic.lower(), "No information found.")

In [6]:
from datetime import datetime

@tool
def format_date(date_string: str) -> str:
    """
    Converts YYYY-MM-DD into a readable format.
    """

    try:
        date = datetime.strptime(date_string, "%Y-%m-%d")
        return date.strftime("%d %B %Y")

    except Exception:
        return "Invalid date format."

In [7]:
@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """
    Converts currency using fixed exchange rates.
    """

    rates = {
        ("USD", "PKR"): 285,
        ("PKR", "USD"): 1 / 285,
        ("USD", "EUR"): 0.85,
        ("EUR", "USD"): 1.18,
    }

    key = (from_currency.upper(), to_currency.upper())

    if key not in rates:
        return "Conversion not supported."

    converted = amount * rates[key]

    return f"{converted:.2f} {to_currency.upper()}"

In [8]:
@tool
def summarize_text(text: str) -> str:
    """
    Returns a simple summary by extracting the first sentence.
    """

    sentences = text.split(".")

    if sentences and sentences[0].strip():
        return sentences[0].strip() + "."

    return "No summary available."

In [9]:
tools = [
    calculate,
    weather,
    search_db,
    format_date,
    convert_currency,
    summarize_text,
]

llm_with_tools = llm.bind_tools(tools)

print("Tools registered successfully!")

Tools registered successfully!


In [10]:
conversation = []

tool_map = {
    "calculate": calculate,
    "weather": weather,
    "search_db": search_db,
    "format_date": format_date,
    "convert_currency": convert_currency,
    "summarize_text": summarize_text,
}

In [11]:
def execute_tool(response):

    if not response.tool_calls:
        return response.content

    tool_call = response.tool_calls[0]

    tool_name = tool_call["name"]
    tool_args = tool_call["args"]

    tool = tool_map[tool_name]

    return tool.invoke(tool_args)

In [16]:
def assistant(user_input):

    conversation.append(HumanMessage(content=user_input))

    response = llm_with_tools.invoke(conversation)

    if response.tool_calls:
        result = execute_tool(response)
    else:
        result = response.content

    conversation.append(AIMessage(content=result))

    print(f"👤 User: {user_input}")
    print(f"🤖 Assistant: {result}\n")

# Conversation Example 1

The assistant remembers user information and stores it in the conversation history.

In [21]:
assistant("Remember that my name is Faizan.")

👤 User: Remember that my name is Faizan.
🤖 Assistant: I remember that your name is Faizan and you study Artificial Intelligence at FAST University.



In [22]:
assistant("Remember that I study Artificial Intelligence at FAST University.")

👤 User: Remember that I study Artificial Intelligence at FAST University.
🤖 Assistant: I remember that your name is Faizan and you study Artificial Intelligence at FAST University.



In [23]:
assistant("What information do you remember about me?")

👤 User: What information do you remember about me?
🤖 Assistant: I remember that your name is Faizan and you study Artificial Intelligence at FAST University.



# Conversation Example 2 – Tool Calling

The assistant can also perform external tasks using the available tools.

In [24]:
assistant("Calculate (250 + 50) * 4")

👤 User: Calculate (250 + 50) * 4
🤖 Assistant: 1200



In [25]:
assistant("What is the weather in Islamabad?")

👤 User: What is the weather in Islamabad?
🤖 Assistant: Sunny, 42°C



In [26]:
assistant("Tell me about LangChain.")

👤 User: Tell me about LangChain.
🤖 Assistant: LangChain is a framework for LLM applications.



# Resetting Memory

Conversation memory can be cleared to start a completely new interaction.

In [27]:
conversation.clear()

print("Conversation memory cleared!")

Conversation memory cleared!


In [28]:
assistant("What is my name?")

👤 User: What is my name?
🤖 Assistant: I don't have any information about your name. This conversation just started, and I don't have the ability to access personal data or recall previous conversations. If you'd like to share your name, I'd be happy to chat with you.



# Advantages of Combining Memory and Tools

By combining conversational memory with external tools, AI assistants become significantly more capable.

Advantages include:

- Context-aware conversations
- Accurate calculations
- Information retrieval
- Better user experience
- Modular system design
- Improved automation

# Limitations

Some limitations of memory-enabled AI assistants include:

- Memory consumes additional tokens.
- Long conversations may increase latency.
- Tool quality depends on implementation.
- External APIs may fail or become unavailable.
- Memory management strategies are required for production systems.

# Observations

During this lab, the following observations were made:

- The assistant successfully remembered previous user information.
- Tool selection was performed automatically by the LLM.
- Custom tools extended the capabilities of the language model.
- Memory improved conversational continuity.
- The modular architecture allows additional tools to be integrated easily.

# Conclusion

In this lab, a Memory-Enabled AI Assistant was developed using LangChain and Groq.

The assistant combined conversational memory with multiple custom tools, allowing it to remember previous interactions while performing calculations, searching information, formatting dates, converting currencies, and summarizing text.

This lab demonstrates how memory and tool calling work together to build intelligent AI assistants capable of handling more realistic and complex conversations.